# Portfolio v7: quantized DFTHC dimension search

The v6 offset-only experiment scored **10.88057**, essentially unchanged from
10.88048. That result closes the offset-only route. This notebook changes the
factorization dimensions and searches the resource discontinuities that the
score actually rewards.

It starts from the fully verified Q82 inverse-square-root, Q898 DQI and R26
chemistry portfolio. The chemistry search contribution-ranks groups, basis
vectors and ensemble coordinates; retrains with the official starter-kit
trainer; quantizes to ten-bit Givens words; freezes those decoded vectors for a
weight refit; and applies the authoritative numerical predicates. Only an
accepted candidate whose exact circuit resources beat the incumbent chemistry
cost can replace it. The final selected portfolio is then verified at full shot
counts and packaged. If every experiment fails, the notebook emits the same
verified score-10 portfolio.

The default `TRAIN_PRESET="full"` runs the near score-11 frontier and the
larger address/QROAM experiments. `"score11"` stops after the R26/B46 and R20
branches for a shorter run.

| Candidate | Why it is included |
|---|---|
| R26/B49/C30 | About 11.02 projected at the incumbent lambda with the public loader |
| R26/B46/C30 or R20/B60/C30 | About 11.15–11.20 projected |
| R19/B49/C24 | Crosses the 1,024-row address boundary; about 11.83 projected |
| R12/B36/C20 | Crosses the inner-QROAM block-16 cliff; about 16.0 projected |

Projections are resource estimates. Only the exact quantized verifier result
shown at the end is a valid competition result.


In [ ]:
import hashlib
import json
import math
import os
import pathlib
import platform
import shutil
import sys
import threading
import time
import zipfile

os.environ.update(
    OMP_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    XLA_PYTHON_CLIENT_PREALLOCATE="false",
)

import numpy as np
import pandas as pd
from IPython.display import display

INPUT = pathlib.Path(os.environ.get("QCC_INPUT", "/kaggle/input"))
WORK = pathlib.Path(os.environ.get("QCC_WORK", "/kaggle/working"))
SUB = pathlib.Path(os.environ.get("QCC_SUB", "/tmp/qcc_submission_v7"))
SLUG = "google-verifiable-quantum-circuit-compilation"
REFERENCE = {
    "invsqrt": 2076774.233715282,
    "dqi": 24439474177519.598,
    "dfthcsos": 3355599588.3155365,
}
PLAN = {
    "invsqrt": {
        "dataset_version_id": 21216223,
        "slug": "qcc-best-invsqrt-circuit",
        "rel": "invsqrt.kmb",
        "sha256": "0f84f5bab3a02e6b357d4656553607174345a33d252cef643f9136db13212141",
    },
    "dqi": {
        "dataset_version_id": 21211427,
        "slug": "qcc-best-dqi",
        "rel": "dqi.kmb",
        "sha256": "c5da9cd4e2d73dd41417e539f2659009d646b3ee30ddb5ca494261378a502856",
    },
    "dfthcsos": {
        "dataset_version_id": 21216621,
        "slug": "qcc-best-circuits",
        "rel": ".",
        "sha256": "47187a5d3664c76e48f1266c39a24b55a7bcc845743d6a673c26c8e69b5cbcee",
    },
}


def sha256_file(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def content_sha256(path):
    path = pathlib.Path(path)
    digest = hashlib.sha256()
    if path.is_dir():
        for child in sorted(item for item in path.rglob("*") if item.is_file()):
            digest.update(child.relative_to(path).as_posix().encode())
            digest.update(child.read_bytes())
    else:
        digest.update(path.read_bytes())
    return digest.hexdigest()


def competition_candidates(root):
    preferred = [root, root / "competitions" / SLUG, root / SLUG]
    discovered = [path.parent for path in root.rglob("kickmix") if path.is_dir()]
    unique = []
    for path in preferred + sorted(discovered):
        path = path.resolve()
        if path not in unique and (path / "kickmix").is_dir() and (path / "verifiers").is_dir():
            unique.append(path)
    return unique


def unique_dataset_root(slug):
    hits = sorted(path.resolve() for path in INPUT.rglob(slug)
                  if path.is_dir() and path.name == slug)
    unique = list(dict.fromkeys(hits))
    if len(unique) != 1:
        raise RuntimeError(f"Expected exactly one {slug!r} dataset directory; found {unique}")
    return unique[0]


if sys.version_info[:2] != (3, 13):
    raise RuntimeError(f"kickmix.so requires CPython 3.13; found {platform.python_version()}")
competition_dirs = competition_candidates(INPUT)
if len(competition_dirs) != 1:
    raise RuntimeError(f"Expected one competition input; found {competition_dirs}")
COMP = competition_dirs[0]

SOURCES = {}
for task, entry in PLAN.items():
    root = unique_dataset_root(entry["slug"])
    source = (root / entry["rel"]).resolve()
    observed = content_sha256(source)
    if observed != entry["sha256"]:
        raise AssertionError(f"{task}: expected {entry['sha256']}, got {observed}")
    SOURCES[task] = source
    print(f"{task:9s} dataset v{entry['dataset_version_id']} hash OK")

starter_hits = sorted(path.resolve() for path in INPUT.rglob("starter_kit")
                      if (path / "dfthc" / "train_df_thc_sga.py").is_file())
starter_hits = list(dict.fromkeys(starter_hits))
if len(starter_hits) != 1:
    raise RuntimeError(f"Expected starter-kit dataset version 21107076; found {starter_hits}")
STARTER = starter_hits[0]

WORK.mkdir(parents=True, exist_ok=True)
shutil.rmtree(SUB, ignore_errors=True)
(SUB / "dfthcsos").mkdir(parents=True)
shutil.copy2(SOURCES["invsqrt"], SUB / "invsqrt.kmb")
shutil.copy2(SOURCES["dqi"], SUB / "dqi.kmb")
chem_source = SOURCES["dfthcsos"]
for source in sorted(path for path in chem_source.iterdir()
                     if path.name == "submission.npz" or
                     (path.name.startswith("angle_k_") and path.suffix == ".kmb")):
    shutil.copy2(source, SUB / "dfthcsos" / source.name)

sys.path[:0] = [str(COMP / "kickmix"), str(COMP / "verifiers"), str(STARTER)]
import kickmix as km

available_cpus = os.cpu_count() or 4
WORKERS = max(1, min(int(os.environ.get("QCC_WORKERS", available_cpus)), available_cpus, 4))
print(f"Competition data: {COMP}")
print(f"Starter kit     : {STARTER}")
print(f"Submission tree : {SUB}")
print(f"Workers / CPUs  : {WORKERS} / {available_cpus}")


In [ ]:
# Install the reviewed source modules inside this run.
SOURCE_ROOT = pathlib.Path("/tmp/qcc_v7_source")
shutil.rmtree(SOURCE_ROOT, ignore_errors=True)
(SOURCE_ROOT / "dfthcsos" / "group_loader").mkdir(parents=True)
(SOURCE_ROOT / "data").mkdir(parents=True)
os.symlink(COMP, SOURCE_ROOT / "data" / "comp", target_is_directory=True)
(SOURCE_ROOT / "dfthcsos" / "group_loader" / "four_address.py").write_text('"""Four-address signed-gauge ANGLE XOR compiler with measured bank erasure.\n\nSPDX-License-Identifier: Apache-2.0\n\nReviewed port of notebook #07, ahmedberatozer/quantum-ai-df-thc-group-compression,\ninternal Version 2, cells 8 (gauge_qrom.py) and 3 (banked_lab.py), supplied\n2026-10-24 snapshot; own Kaggle version ID unknown. Apache-2.0 code licence\nrecorded in public/READING.md. Retained lineage: Kozykorpesh / Sumit Chongder\nbanked QROM, through Kirill Tym\'s quantum-circuits-qrom-tower42 v10 and Artem\n(anazemcev)\'s truncated-ANF and split-ANF eraser. Public source was read as\nuntrusted text, never imported or executed. No numerical checkpoint is copied.\n\nAll public targets may be arbitrary. Only the private bank is initialized\nclean. Every illegal high-address value has the same polynomial completion\nin initialization and measured erasure, so cleanup is not dependent on an\nunstated zero-padding assumption.\n"""\nfrom __future__ import annotations\nfrom functools import lru_cache\nfrom pathlib import Path\nimport sys\n\nimport numpy as np\nROOT = Path(__file__).resolve().parents[2]\nsys.path.insert(0,str(ROOT/\'data/comp/kickmix\'))\nimport kickmix as km\n\n\ndef mobius(values):\n    out=list(map(int,values))\n    if not out or len(out)&(len(out)-1):\n        raise ValueError(\'GF(2) transform length must be a power of two\')\n    for bit in range((len(out)-1).bit_length()):\n        step=1<<bit\n        for lo in range(0,len(out),2*step):\n            for offset in range(step):\n                out[lo+step+offset]^=out[lo+offset]\n    return out\n\n\ndef bits(value):\n    while value:\n        one=value&-value\n        yield one.bit_length()-1\n        value^=one\n\n\ndef xor_product(builder,left,right,targets):\n    """One CCX fans out a product into arbitrary targets, with no clean pivot."""\n    targets=list(targets)\n    if not targets:return\n    pivot=targets[0]\n    for target in targets[1:]:builder.cx(pivot,target)\n    builder.ccx(left,right,pivot)\n    for target in reversed(targets[1:]):builder.cx(pivot,target)\n\n\ndef initialize_forest(builder,table,address,outputs,scratch):\n    """Simple fixed forest when scratch holds every non-leaf prefix product."""\n    address,outputs,scratch=map(list,(address,outputs,scratch))\n    count,n=len(table),len(address)\n    coeff=mobius(list(table)+[0]*((1<<n)-count))\n    coeff[count:]=[0]*((1<<n)-count)\n    completed=mobius(coeff)\n    if completed[:count]!=list(table):raise AssertionError(\'prefix completion changed a legal row\')\n    stats=dict(interior_products=0,recomputed_products=0,pebble_sites=0,\n               terminal_products=0,scratch_qubits=len(scratch),\n               high_rows=count,high_address_bits=n,mode=\'fixed forest\')\n\n    def children(mask):\n        return [(mask|(1<<bit),bit) for bit in range(mask.bit_length(),n)\n                if (mask|(1<<bit))<count]\n\n    def targets(mask):return [outputs[bit] for bit in bits(coeff[mask])]\n\n    def walk(mask,control):\n        ts=targets(mask)\n        if ts:\n            if control is None:builder.x(km.array(ts))\n            else:builder.cx(control,km.array(ts))\n        for child,bit in children(mask):\n            if mask==0:walk(child,address[bit])\n            elif not children(child):\n                ts=targets(child)\n                if ts:\n                    xor_product(builder,control,address[bit],ts)\n                    stats[\'terminal_products\']+=1\n            else:\n                product=scratch[child.bit_count()-2]\n                builder.init_and(control,address[bit],product)\n                stats[\'interior_products\']+=1\n                walk(child,product)\n                builder.del_and(control,address[bit],product)\n\n    walk(0,None)\n    return completed,stats\n\n\ndef initialize_pebbled(builder,table,address,outputs,scratch):\n    """Prefix ANF forest, terminal target leaves and safe ancestor pebbling."""\n    address,outputs,scratch=map(list,(address,outputs,scratch))\n    count,n=len(table),len(address)\n    if not 1<count<=1<<n or count<=1<<(n-1):\n        raise ValueError(\'table must be a prefix using every high-address bit\')\n    coeff=mobius(list(table)+[0]*((1<<n)-count))\n    coeff[count:]=[0]*((1<<n)-count)\n    completed=mobius(coeff)\n    if completed[:count]!=list(table):\n        raise AssertionError(\'triangular completion changed a legal row\')\n    stats=dict(interior_products=0,recomputed_products=0,pebble_sites=0,\n               terminal_products=0,scratch_qubits=len(scratch),\n               high_rows=count,high_address_bits=n,mode=\'ancestor pebbled\')\n\n    @lru_cache(None)\n    def children(mask):\n        return tuple((mask|(1<<bit),bit) for bit in range(mask.bit_length(),n)\n                     if (mask|(1<<bit))<count)\n\n    @lru_cache(None)\n    def needed(mask):\n        return max([1+needed(child) for child,_ in children(mask) if children(child)]+[0])\n\n    def targets(mask):return [outputs[bit] for bit in bits(coeff[mask])]\n\n    def walk(mask,control,path,free):\n        erased=[]\n        if mask.bit_count()==5 and needed(mask)>len(free):\n            shortage=needed(mask)-len(free)\n            if not 0<shortage<=len(path)-1:\n                raise ValueError(\'five-scratch pebbling schedule cannot cover this table\')\n            erased=path[:shortage]\n            for left,right,target in reversed(erased):\n                builder.del_and(left,right,target)\n                free.append(target)\n            stats[\'pebble_sites\']+=1\n        ts=targets(mask)\n        if ts:\n            if control is None:builder.x(km.array(ts))\n            else:builder.cx(control,km.array(ts))\n        for child,bit in children(mask):\n            if mask==0:\n                walk(child,address[bit],path,free)\n            elif not children(child):\n                ts=targets(child)\n                if ts:\n                    xor_product(builder,control,address[bit],ts)\n                    stats[\'terminal_products\']+=1\n            else:\n                if not free:raise ValueError(\'scratch exhausted in prefix forest\')\n                target=free.pop()\n                builder.init_and(control,address[bit],target)\n                stats[\'interior_products\']+=1\n                walk(child,target,path+[(control,address[bit],target)],free)\n                builder.del_and(control,address[bit],target)\n                free.append(target)\n        for left,right,target in erased:\n            free.remove(target)\n            builder.init_and(left,right,target)\n            stats[\'recomputed_products\']+=1\n\n    walk(0,None,[],scratch.copy())\n    return completed,stats\n\n\ndef erase_lookup(builder,table,address,target,low_bits=5):\n    """X-measure a clean lookup bank and exactly cancel its measurement phase.\n\n    Reset bank qubits are reused for low-half monomials and high-half carries.\n    Their own del_and calls correct all newly introduced HMR phases.\n    """\n    address,target=map(list,(address,target))\n    n=len(address)\n    if len(table)!=1<<n or not 0<low_bits<n:\n        raise ValueError(\'eraser needs a full table and two nonempty address halves\')\n    if any(value<0 or value>=1<<len(target) for value in table):\n        raise ValueError(\'lookup value exceeds private bank width\')\n    coeff=mobius(table)\n    if mobius(coeff)!=list(table):raise AssertionError(\'eraser ANF identity failed\')\n    measurements=builder.alloc_bits(len(target))\n    builder.hmr(km.array(target),measurements)\n    parity=builder.alloc_bits(1)[0]\n    high_bits=n-low_bits\n    low_masks=[mask for mask in range(1,1<<low_bits) if mask.bit_count()>1]\n    needed=len(low_masks)+high_bits-1\n    extra=builder.alloc_qubits(max(0,needed-len(target)))\n    workspace=target+list(extra)\n    low={1<<bit:address[bit] for bit in range(low_bits)}\n    for mask,qubit in zip(low_masks,workspace):\n        last=mask.bit_length()-1\n        builder.init_and(low[mask^(1<<last)],address[last],qubit)\n        low[mask]=qubit\n    carries=workspace[len(low_masks):needed]\n\n    def correct(high_mask,high_qubit):\n        for low_mask in range(1<<low_bits):\n            coefficient=coeff[low_mask|(high_mask<<low_bits)]\n            if not coefficient:continue\n            builder.bit_store(parity,False)\n            for bit in bits(coefficient):builder.cx(measurements[bit],parity)\n            if not low_mask and not high_mask:\n                builder.neg_if(parity)\n            else:\n                builder.push_condition(parity)\n                if not low_mask:builder.z(high_qubit)\n                elif not high_mask:builder.z(low[low_mask])\n                else:builder.cz(low[low_mask],high_qubit)\n                builder.pop_condition()\n\n    def walk(mask,control):\n        correct(mask,control)\n        for bit in range(mask.bit_length(),high_bits):\n            child=mask|(1<<bit)\n            if not mask:walk(child,address[low_bits+bit])\n            else:\n                product=carries[mask.bit_count()-1]\n                builder.init_and(control,address[low_bits+bit],product)\n                walk(child,product)\n                builder.del_and(control,address[low_bits+bit],product)\n\n    walk(0,None)\n    for mask in reversed(low_masks):\n        last=mask.bit_length()-1\n        builder.del_and(low[mask^(1<<last)],address[last],low[mask])\n    builder.free(extra)\n    return dict(high_address_bits=n,low_bits=low_bits,other_bits=high_bits,\n                retained_products=len(low_masks),traversal_carries=high_bits-1,\n                scratch_qubits=needed,extra_qubits=len(extra),\n                toffoli=(1<<low_bits)-low_bits-1+(1<<high_bits)-high_bits-1)\n\n\ndef tables(delta):\n    """Return full base, three compact differences, and exact four-case certificate."""\n    delta=np.asarray(delta)\n    if delta.ndim!=2 or delta.shape[1]!=3 or len(delta)<8 or len(delta)%4:\n        raise ValueError(\'four-address compiler requires (A,3), A>=8 divisible by four\')\n    if not np.issubdtype(delta.dtype,np.integer) or np.any(delta<0) or np.any(delta>=1024):\n        raise ValueError(\'delta words must be ten-bit integers\')\n    if np.any(((delta>>9)^(delta>>8))&1):\n        raise ValueError(\'ten-bit signed half-open gauge requires bit9=bit8\')\n    packed=[sum(int(value)<<(10*j) for j,value in enumerate(row)) for row in delta]\n    compact=lambda value:sum(((value>>(10*j))&511)<<(9*j) for j in range(3))\n    expand=lambda value:sum((((value>>(9*j))&511)|(((value>>(9*j+8))&1)<<9))<<(10*j) for j in range(3))\n    base=[];diff=[[],[],[]]\n    for lo in range(0,len(packed),4):\n        a,b,c,d=packed[lo:lo+4]\n        base.append(a)\n        diff[0].append(compact(a^b))\n        diff[1].append(compact(a^c))\n        diff[2].append(compact(a^b^c^d))\n    restored=[base[h]^expand((diff[0][h] if case&1 else 0)^\n                            (diff[1][h] if case&2 else 0)^\n                            (diff[2][h] if case==3 else 0))\n              for h in range(len(base)) for case in range(4)]\n    if restored!=packed:raise AssertionError(\'four-address integer certificate failed\')\n    values=[base[h]|(diff[0][h]<<30)|(diff[1][h]<<57)|(diff[2][h]<<84)\n            for h in range(len(base))]\n    return values,expand\n\n\ndef build(delta,brot=10,*,return_report=False,scratch_qubits=None):\n    if brot!=10:raise ValueError(\'four-address compiler requires brot10\')\n    delta=np.asarray(delta)\n    values,_=tables(delta)\n    n=(len(delta)-1).bit_length()\n    if n-2<=5:raise ValueError(\'this eraser requires more than five high-address bits\')\n    if scratch_qubits is None:scratch_qubits=6 if n+30+81+6<=128 else 5\n    if scratch_qubits not in (5,6) or n+30+81+scratch_qubits>128:\n        raise ValueError(\'scratch choice does not fit the128-qubit circuit cap\')\n    builder=km.CircuitBuilder()\n    selection=builder.create_quantum_register(n,\'selection\')\n    output=[qubit for j in range(3) for qubit in builder.create_quantum_register(10,f\'word_{j}\')]\n    bank=list(builder.alloc_qubits(81))\n    scratch=builder.alloc_qubits(scratch_qubits)\n    high_bits=n-2\n    need=max((mask.bit_count()-1 for mask in range(1,len(values))\n              if any((mask|(1<<bit))<len(values) for bit in range(mask.bit_length(),high_bits))),default=0)\n    initializer=initialize_forest if need<=scratch_qubits else initialize_pebbled\n    completed,init=initializer(builder,values,selection[2:],output+bank,scratch)\n    builder.free(scratch)\n    both=builder.alloc_qubits(1)[0]\n    builder.init_and(selection[0],selection[1],both)\n    for s,control in enumerate((selection[0],selection[1],both)):\n        for word in range(3):\n            for bit in range(9):\n                targets=[output[word*10+bit]]\n                if bit==8:targets.append(output[word*10+9])\n                xor_product(builder,control,bank[s*27+word*9+bit],targets)\n    builder.del_and(selection[0],selection[1],both)\n    builder.free(both)\n    erasure=erase_lookup(builder,[value>>30 for value in completed],selection[2:],bank,low_bits=5)\n    builder.free(km.array(bank))\n    circuit=builder.finish_circuit()\n    expected=init[\'interior_products\']+init[\'recomputed_products\']+init[\'terminal_products\']+82+erasure[\'toffoli\']\n    actual=int(circuit.operation_statistics().num_toffoli_like)\n    if actual!=expected:raise AssertionError(\'native circuit and derived operation counts differ\')\n    if circuit.num_qubits>128:raise ValueError(\'four-address circuit exceeds128 qubits\')\n    report=dict(rows=len(delta),brot=10,J=3,selection_bits=n,num_qubits=circuit.num_qubits,\n                bank_qubits=81,initialization_scratch=scratch_qubits,initialization=init,erasure=erasure,\n                controlled_difference_products=81,low_address_product=1,\n                toffoli_per_call=actual,integer_table_identity=\'PASS\')\n    return (circuit,report) if return_report else circuit\n')
(SOURCE_ROOT / "dfthcsos_dimension_train.py").write_text('#!/usr/bin/env python3\n"""Utilities for dimension-aware DFTHC-SOS pruning and candidate selection.\n\nThe numerical trainer itself is supplied by the official Kaggle starter kit.\nThis module contains the deterministic parts of the v7 search: contribution\nranking, shape reduction, strict metric checks and incumbent comparison.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport math\nfrom typing import Any, Mapping\n\nimport numpy as np\n\n\nREQUIRED_PARAMETER_KEYS = ("U", "W", "H2_I", "H1_I", "Hsym_lower")\n\n\ndef numpy_parameters(params: Mapping[str, Any]) -> dict[str, np.ndarray]:\n    """Copy the five submitted arrays into ordinary native NumPy arrays."""\n    missing = [name for name in REQUIRED_PARAMETER_KEYS if name not in params]\n    if missing:\n        raise KeyError(f"missing factorization arrays: {missing}")\n    result = {name: np.asarray(params[name]).copy() for name in REQUIRED_PARAMETER_KEYS}\n    if result["U"].ndim != 3 or result["W"].ndim != 3 or result["H2_I"].ndim != 2:\n        raise ValueError("expected U(R,B,N), W(R,B,C), and H2_I(R,C)")\n    r, b, _ = result["U"].shape\n    if result["W"].shape[:2] != (r, b):\n        raise ValueError("U and W disagree on (R,B)")\n    if result["H2_I"].shape != (r, result["W"].shape[2]):\n        raise ValueError("H2_I and W disagree on (R,C)")\n    return result\n\n\ndef _descending_indices(score: np.ndarray, count: int, axis: int) -> np.ndarray:\n    if not 0 < count <= score.shape[axis]:\n        raise ValueError(f"cannot retain {count} entries along shape {score.shape}")\n    # Stable sorting makes checkpoint production deterministic when scores tie.\n    return np.flip(np.argsort(score, axis=axis, kind="stable"), axis=axis).take(\n        indices=range(count), axis=axis\n    )\n\n\ndef prune_dimensions(\n    params: Mapping[str, Any],\n    rank_r: int,\n    bases_b: int,\n    ensemble_c: int,\n) -> tuple[dict[str, np.ndarray], dict[str, Any]]:\n    """Reduce an existing factorization using contribution-ranked slices.\n\n    Groups are ranked globally.  Basis and ensemble coordinates are ranked\n    independently inside every retained group.  Reordering is legal because\n    those axes are summed by the verifier.  Training must still repair the\n    numerical approximation; this function makes no acceptance claim.\n    """\n    source = numpy_parameters(params)\n    u, w, h2 = source["U"], source["W"], source["H2_I"]\n    r0, b0, c0 = w.shape\n    if rank_r > r0 or bases_b > b0 or ensemble_c > c0:\n        raise ValueError(\n            f"target {(rank_r, bases_b, ensemble_c)} must not exceed source {(r0, b0, c0)}"\n        )\n\n    # U vectors are close to unit length, but include their measured norm so\n    # this also behaves sensibly on intermediate, unnormalised checkpoints.\n    # The verifier stores H2_I but uses the identity offset\n    # W_offset = H2_I - sum_b(W).  Preserve that offset exactly while basis\n    # entries are removed; keeping H2_I itself would inject the removed basis\n    # weights into every retained offset and gives the trainer a poor start.\n    offset = h2 - np.sum(w, axis=1)\n    u_norm = np.linalg.norm(u, axis=2)\n    basis_score = np.sum(np.abs(w), axis=2) * np.maximum(u_norm, 1e-12)\n    group_score = np.sum(basis_score, axis=1) + np.sum(np.abs(offset), axis=1)\n    group_index = _descending_indices(group_score, rank_r, axis=0)\n\n    u = u[group_index]\n    w = w[group_index]\n    offset = offset[group_index]\n    basis_score = basis_score[group_index]\n    basis_index = _descending_indices(basis_score, bases_b, axis=1)\n    u = np.take_along_axis(u, basis_index[:, :, None], axis=1)\n    w = np.take_along_axis(w, basis_index[:, :, None], axis=1)\n\n    ensemble_score = np.sum(np.abs(w), axis=1) + np.abs(offset)\n    ensemble_index = _descending_indices(ensemble_score, ensemble_c, axis=1)\n    w = np.take_along_axis(w, ensemble_index[:, None, :], axis=2)\n    offset = np.take_along_axis(offset, ensemble_index, axis=1)\n    h2 = offset + np.sum(w, axis=1)\n\n    reduced = {\n        "U": np.ascontiguousarray(u),\n        "W": np.ascontiguousarray(w),\n        "H2_I": np.ascontiguousarray(h2),\n        "H1_I": source["H1_I"],\n        "Hsym_lower": source["Hsym_lower"],\n    }\n    record = {\n        "source_shape": [r0, b0, c0],\n        "target_shape": [rank_r, bases_b, ensemble_c],\n        "group_index": group_index.tolist(),\n        "basis_index": basis_index.tolist(),\n        "ensemble_index": ensemble_index.tolist(),\n        "retained_group_score_fraction": float(\n            np.sum(group_score[group_index]) / max(np.sum(group_score), 1e-300)\n        ),\n    }\n    return reduced, record\n\n\ndef accepted_metrics(metrics: Mapping[str, Any]) -> bool:\n    """Mirror the verifier\'s strict numerical acceptance predicates."""\n    return bool(\n        float(metrics["error_1"]) <= 0.020\n        and float(metrics["error_2"]) <= 0.020\n        and float(metrics["Lambda"]) > 49.0\n        and float(metrics["E_gap"]) > 3.914898\n        and bool(metrics["gap_in_domain"])\n        and math.isfinite(float(metrics["lambda_eff"]))\n    )\n\n\n@dataclass(frozen=True)\nclass CandidateCost:\n    shape: tuple[int, int, int]\n    lambda_eff: float\n    toffoli: float\n    qubits: int\n    cost: float\n    accepted: bool\n\n    @property\n    def beats_incumbent(self) -> bool:\n        return self.accepted and math.isfinite(self.cost)\n\n\ndef candidate_cost(\n    shape: tuple[int, int, int],\n    metrics: Mapping[str, Any],\n    resources: Mapping[str, Any],\n    *,\n    incumbent: float | None = None,\n) -> CandidateCost:\n    accepted = accepted_metrics(metrics) and int(resources["angles_qubits"]) <= 128\n    value = (\n        float(metrics["lambda_eff"])\n        * float(resources["toffoli"])\n        * int(resources["qubits"])\n        if accepted\n        else math.inf\n    )\n    if incumbent is not None and not value < incumbent:\n        accepted = False\n    return CandidateCost(\n        shape=tuple(map(int, shape)),\n        lambda_eff=float(metrics["lambda_eff"]),\n        toffoli=float(resources["toffoli"]),\n        qubits=int(resources["qubits"]),\n        cost=float(value),\n        accepted=accepted,\n    )\n\n\ndef self_test() -> None:\n    rng = np.random.default_rng(20261026)\n    params = {\n        "U": rng.normal(size=(5, 7, 11)),\n        "W": rng.normal(size=(5, 7, 6)),\n        "H2_I": rng.normal(size=(5, 6)),\n        "H1_I": rng.normal(size=(11, 11)),\n        "Hsym_lower": rng.normal(size=(11, 11)),\n    }\n    a, record = prune_dimensions(params, 4, 5, 3)\n    b, record2 = prune_dimensions(params, 4, 5, 3)\n    assert a["U"].shape == (4, 5, 11)\n    assert a["W"].shape == (4, 5, 3)\n    assert a["H2_I"].shape == (4, 3)\n    assert all(np.array_equal(a[name], b[name]) for name in REQUIRED_PARAMETER_KEYS)\n    assert record == record2\n    original_offset = params["H2_I"] - np.sum(params["W"], axis=1)\n    expected_offset = np.take_along_axis(\n        original_offset[np.asarray(record["group_index"])],\n        np.asarray(record["ensemble_index"]),\n        axis=1,\n    )\n    actual_offset = a["H2_I"] - np.sum(a["W"], axis=1)\n    assert np.allclose(actual_offset, expected_offset)\n\n\nif __name__ == "__main__":\n    self_test()\n    print("dimension-pruning self-test: PASS")\n')
(SOURCE_ROOT / "dfthcsos_offset_refit.py").write_text('#!/usr/bin/env python3\n"""Verifier-aligned fixed-word offset refit for a DFTHC-SOS submission.\n\nThe ANGLE circuits and all ``W`` tensor entries stay fixed.  Only the stored\n``H2_I`` array is changed, which changes the verifier\'s offset\n``W_offset = H2_I - sum_b(W)``.  Candidate selection always uses the official\n``evaluate_words`` and ``passes_accuracy`` functions.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, replace\nimport math\nfrom pathlib import Path\nimport sys\nfrom typing import Any\n\nimport numpy as np\nfrom scipy.optimize import minimize\n\n\n@dataclass\nclass OffsetModel:\n    """Fast lambda objective and analytic gradient at fixed decoded vectors."""\n\n    h0: np.ndarray\n    x: np.ndarray\n    w: np.ndarray\n    beta_1: float\n    ecore: float\n    electrons: int\n    energy_upper: float\n    start: np.ndarray\n\n    def __post_init__(self) -> None:\n        self.x = np.asarray(self.x, dtype=np.float64)\n        self.w = np.asarray(self.w, dtype=np.float64)\n        self.start = np.asarray(self.start, dtype=np.float64)\n        self.sum_w = np.sum(self.w, axis=1)\n        self.l1_w = np.linalg.norm(self.w, ord=1, axis=1)\n        self.trace_x = np.trace(self.x, axis1=-2, axis2=-1)\n\n    def value_gradient(\n        self,\n        flat: np.ndarray,\n        *,\n        gap_floor: float = 3.914899,\n        gap_margin: float = 2e-6,\n        floor_penalty: float = 2e6,\n        trust_penalty: float = 1e-7,\n    ) -> tuple[float, np.ndarray]:\n        offsets = np.asarray(flat, dtype=np.float64).reshape(self.start.shape)\n        h1_prime = self.h0 - np.einsum("rc,rcpq->pq", offsets, self.x, optimize=True)\n        eigenvalues, eigenvectors = np.linalg.eigh(h1_prime)\n        signs = np.sign(eigenvalues)\n        sign_h = (eigenvectors * signs[None, :]) @ eigenvectors.T\n        overlap = np.einsum("pq,rcpq->rc", sign_h, self.x, optimize=True)\n\n        lambda_h1 = float(np.sum(np.abs(eigenvalues)))\n        w_b = offsets + self.sum_w\n        widths = self.l1_w + np.abs(w_b)\n        lambda_h2 = 0.25 * float(np.sum(widths * widths))\n        Lambda = lambda_h1 + lambda_h2\n\n        trace_h1 = float(np.sum(eigenvalues))\n        e_sos = (\n            self.ecore\n            - self.electrons * self.beta_1\n            - (lambda_h1 - trace_h1)\n            - 0.5 * float(np.sum(offsets * offsets))\n        )\n        e_gap = self.energy_upper - e_sos\n        radicand = e_gap * (2.0 * Lambda - e_gap)\n        if not np.isfinite(radicand) or radicand <= 0:\n            # A large smooth-ish rejection value keeps L-BFGS-B inside domain.\n            delta = offsets - self.start\n            return 1e9 + float(np.sum(delta * delta)), (2.0 * delta).ravel()\n\n        lambda_eff = math.sqrt(radicand)\n        grad_lambda_h1 = -overlap\n        grad_lambda_h2 = 0.5 * widths * np.sign(w_b)\n        grad_Lambda = grad_lambda_h1 + grad_lambda_h2\n        grad_gap = -overlap + self.trace_x + offsets\n        gradient = (\n            (Lambda - e_gap) * grad_gap + e_gap * grad_Lambda\n        ) / lambda_eff\n\n        objective = lambda_eff\n        minimum_gap = gap_floor + gap_margin\n        if e_gap < minimum_gap:\n            shortfall = minimum_gap - e_gap\n            objective += floor_penalty * shortfall * shortfall\n            gradient += -2.0 * floor_penalty * shortfall * grad_gap\n        if Lambda <= 49.000001:\n            shortfall = 49.000001 - Lambda\n            objective += floor_penalty * shortfall * shortfall\n            gradient += -2.0 * floor_penalty * shortfall * grad_Lambda\n\n        delta = offsets - self.start\n        objective += trust_penalty * float(np.sum(delta * delta))\n        gradient += 2.0 * trust_penalty * delta\n        return float(objective), gradient.ravel()\n\n\ndef _build_model(inputs: Any, words: np.ndarray, edges: np.ndarray, brot: int) -> OffsetModel:\n    from dfthcsos.scoreinfra.dfthc_math import reconstruct_u_vectors, reconstruct_x, split_u_vectors\n\n    vectors = reconstruct_u_vectors(words / (1 << (brot - 1)), edges)\n    norms = np.linalg.norm(vectors, axis=1, keepdims=True)\n    one_body, two_body = split_u_vectors(vectors / norms, inputs.w)\n    del one_body\n    x = reconstruct_x(two_body, inputs.w)\n    x = np.tril(x) + np.swapaxes(np.tril(x, k=-1), -1, -2)\n    tei = np.asarray(inputs.ref["tei"], dtype=np.float64)\n    oei = np.asarray(inputs.ref["oei"], dtype=np.float64)\n    h1_thc = oei - 0.5 * np.einsum("prrq->pq", tei)\n    h0 = (\n        h1_thc\n        - 0.5 * inputs.ref["electrons"] * inputs.S\n        + inputs.beta_1 * np.eye(inputs.num_orbitals)\n    )\n    return OffsetModel(\n        h0=h0,\n        x=x,\n        w=inputs.w,\n        beta_1=inputs.beta_1,\n        ecore=float(inputs.ref["ecore"]),\n        electrons=int(inputs.ref["electrons"]),\n        energy_upper=float(inputs.energy_upper),\n        start=inputs.W,\n    )\n\n\ndef _serializable(metrics: dict[str, Any]) -> dict[str, Any]:\n    return {\n        key: value.item() if isinstance(value, np.generic) else value\n        for key, value in metrics.items()\n    }\n\n\ndef refit_offsets(\n    factorization: dict[str, np.ndarray],\n    words: np.ndarray,\n    edges: np.ndarray,\n    brot: int,\n    *,\n    radii: tuple[float, ...] = (2e-4, 6e-4, 2e-3, 6e-3, 2e-2),\n    maxiter: int = 80,\n) -> tuple[dict[str, np.ndarray], dict[str, Any]]:\n    """Return the best officially accepted H2_I-only refit and its report."""\n    from dfthcsos.verify import evaluate_words, load_inputs, passes_accuracy\n\n    original = {name: np.array(value) for name, value in factorization.items()}\n    inputs = load_inputs(original)\n    baseline_metrics = evaluate_words(inputs, words, edges, brot)\n    if not passes_accuracy(baseline_metrics):\n        raise ValueError(f"baseline factorization is not accepted: {baseline_metrics}")\n\n    model = _build_model(inputs, words, edges, brot)\n    start = np.array(inputs.W, dtype=np.float64)\n    best_params = original\n    best_metrics = baseline_metrics\n    trials: list[dict[str, Any]] = []\n\n    for radius in radii:\n        widths = radius * (1.0 + np.abs(start))\n        bounds = list(zip((start - widths).ravel(), (start + widths).ravel()))\n\n        def function(flat: np.ndarray) -> tuple[float, np.ndarray]:\n            return model.value_gradient(flat)\n\n        result = minimize(\n            function,\n            start.ravel(),\n            method="L-BFGS-B",\n            jac=True,\n            bounds=bounds,\n            options={"maxiter": maxiter, "ftol": 1e-13, "gtol": 1e-9, "maxls": 30},\n        )\n        offsets = np.asarray(result.x).reshape(start.shape)\n        candidate = {name: np.array(value) for name, value in original.items()}\n        candidate["H2_I"] = offsets + np.sum(inputs.w, axis=1)\n        candidate_inputs = replace(inputs, params=candidate, W=offsets)\n        metrics = evaluate_words(candidate_inputs, words, edges, brot)\n        accepted = bool(passes_accuracy(metrics))\n        improved = accepted and float(metrics["lambda_eff"]) < float(best_metrics["lambda_eff"])\n        trials.append({\n            "radius": radius,\n            "optimizer_success": bool(result.success),\n            "optimizer_status": int(result.status),\n            "optimizer_message": str(result.message),\n            "iterations": int(result.nit),\n            "evaluations": int(result.nfev),\n            "accepted": accepted,\n            "improved": improved,\n            "max_offset_change": float(np.max(np.abs(offsets - start))),\n            "metrics": _serializable(metrics),\n        })\n        if improved:\n            best_params = candidate\n            best_metrics = metrics\n\n    report = {\n        "method": "fixed decoded words; H2_I/offset refit only; official metric selection",\n        "baseline_metrics": _serializable(baseline_metrics),\n        "selected_metrics": _serializable(best_metrics),\n        "improved": float(best_metrics["lambda_eff"]) < float(baseline_metrics["lambda_eff"]),\n        "trials": trials,\n    }\n    return best_params, report\n\n\ndef gradient_check(seed: int = 20261025) -> float:\n    """Finite-difference check of the analytic objective gradient on a small model."""\n    rng = np.random.default_rng(seed)\n    r, c, n, b = 2, 3, 5, 4\n    x = rng.normal(scale=0.05, size=(r, c, n, n))\n    x = 0.5 * (x + np.swapaxes(x, -1, -2))\n    model = OffsetModel(\n        h0=np.diag(np.linspace(-2.0, 2.0, n)) + rng.normal(scale=0.01, size=(n, n)),\n        x=x,\n        w=rng.normal(scale=0.03, size=(r, b, c)),\n        beta_1=0.0,\n        ecore=-10.0,\n        electrons=4,\n        energy_upper=-7.0,\n        start=rng.normal(scale=0.05, size=(r, c)),\n    )\n    model.h0 = 0.5 * (model.h0 + model.h0.T)\n    point = model.start.ravel() + rng.normal(scale=1e-3, size=r * c)\n    _, analytic = model.value_gradient(\n        point, gap_floor=-1e6, floor_penalty=0.0, trust_penalty=0.0\n    )\n    numeric = np.zeros_like(point)\n    step = 1e-6\n    for index in range(len(point)):\n        plus, minus = point.copy(), point.copy()\n        plus[index] += step\n        minus[index] -= step\n        fp = model.value_gradient(plus, gap_floor=-1e6, floor_penalty=0.0, trust_penalty=0.0)[0]\n        fm = model.value_gradient(minus, gap_floor=-1e6, floor_penalty=0.0, trust_penalty=0.0)[0]\n        numeric[index] = (fp - fm) / (2 * step)\n    return float(np.max(np.abs(analytic - numeric)))\n\n\nif __name__ == "__main__":\n    error = gradient_check()\n    print(f"maximum gradient error: {error:.3e}")\n    if error > 2e-5:\n        raise SystemExit(1)\n')
sys.path[:0] = [str(SOURCE_ROOT), str(SOURCE_ROOT / "dfthcsos" / "group_loader")]

import dfthcsos_dimension_train as DT
import dfthcsos_offset_refit as OR
import four_address as FOUR
DT.self_test()
assert OR.gradient_check() < 2e-5

from dfthc.train_df_thc_sga import Initialize, Train
from dfthc import df_thc_sga
from starter.task2_angles.encode import encode_inputs
from starter.task2_angles.trees import log_depth_tree_edges
from verifier.score import load_inputs as kit_load_inputs
import jax
import jax.numpy as jnp
from dfthcsos import verify as VC
from dfthcsos.example_submission.angle_k_reference_circuit import batch_deltas
from dfthcsos.scoreinfra.dfthc_math import reconstruct_u_vectors

print("JAX backend:", jax.default_backend(), jax.devices())
print("Loading the official FeMoCo reference", flush=True)
REFERENCE_DATA = Initialize(str(COMP / "verifiers" / "dfthcsos" / "FeMoCo76o113e.fcidump"),
                            -22140.27335)
TREE_EDGES = np.asarray(log_depth_tree_edges(76), dtype=np.int64)


def load_circuit(path):
    return km.Circuit.from_file(str(path), format="kmb")


BASE_CIRCUITS = [load_circuit(path) for path in sorted(
    (SUB / "dfthcsos").glob("angle_k_*.kmb"),
    key=lambda path: int(path.stem.rsplit("_", 1)[1]),
)]
with np.load(SUB / "dfthcsos" / "submission.npz", allow_pickle=False) as archive:
    BASE_ARCHIVE = {name: np.array(archive[name]) for name in archive.files}
base_edges = np.asarray(BASE_ARCHIVE.pop("tree_edges"))
if not np.array_equal(base_edges, TREE_EDGES):
    raise AssertionError("verified R26 checkpoint uses an unexpected Givens tree")
base_inputs = VC.load_inputs(BASE_ARCHIVE)
base_quick = VC.evaluate_circuits(base_inputs, BASE_CIRCUITS, base_edges,
                                  shots_per_address=1, seed=20261026)
BASE_WORDS = np.asarray(base_quick["angle_words"])
BASE_BROT = int(base_quick["brot"])

# Retain the small v6 offset improvement before using the checkpoint as a seed.
BASE_PARAMETERS, BASE_OFFSET_REPORT = OR.refit_offsets(
    BASE_ARCHIVE, BASE_WORDS, base_edges, BASE_BROT,
    radii=(2e-4, 6e-4, 2e-3, 6e-3, 2e-2), maxiter=80,
)
BASE_PARAMETERS = DT.numpy_parameters(BASE_PARAMETERS)
base_metrics = BASE_OFFSET_REPORT["selected_metrics"]
INCUMBENT_CHEM_COST = VC.score_from_metrics(base_metrics, base_quick["resources"])
if BASE_OFFSET_REPORT["improved"]:
    baseline_payload = {name: np.asarray(value) for name, value in BASE_PARAMETERS.items()}
    baseline_payload["tree_edges"] = base_edges
    baseline_payload["angle_words"] = BASE_WORDS
    np.savez(SUB / "dfthcsos" / "submission.npz", **baseline_payload)
print(f"Chemistry incumbent after guarded offset refit: {INCUMBENT_CHEM_COST:.9g}")


In [ ]:
# Set to "score11" to omit the two long high-upside branches.
TRAIN_PRESET = "full"

SEARCH_PLAN = [
    dict(name="b56", shape=(26, 56, 30), parent="baseline", epochs=(8000, 12000)),
    dict(name="b52", shape=(26, 52, 30), parent="b56", epochs=(10000, 14000)),
    dict(name="b49", shape=(26, 49, 30), parent="b52", epochs=(12000, 18000)),
    dict(name="b46", shape=(26, 46, 30), parent="b49", epochs=(14000, 20000)),
    dict(name="r24", shape=(24, 60, 30), parent="baseline", epochs=(10000, 14000)),
    dict(name="r22", shape=(22, 60, 30), parent="r24", epochs=(12000, 18000)),
    dict(name="r20", shape=(20, 60, 30), parent="r22", epochs=(14000, 22000)),
]
if TRAIN_PRESET == "full":
    SEARCH_PLAN += [
        dict(name="a1024", shape=(19, 49, 24), parent="r20", epochs=(22000, 30000)),
        dict(name="qroam16", shape=(12, 36, 20), parent="a1024", epochs=(32000, 45000)),
    ]
elif TRAIN_PRESET != "score11":
    raise ValueError(f"unknown TRAIN_PRESET={TRAIN_PRESET!r}")


def native(params):
    return {name: np.asarray(value) for name, value in DT.numpy_parameters(params).items()}


def trainer_arrays(params):
    return {name: jnp.asarray(value, dtype=df_thc_sga.dtype)
            for name, value in native(params).items()}


def train_phase(params, *, epochs, rho, learning_rate, learning_rate_final,
                freeze_u=False, label="phase"):
    p = trainer_arrays(params)
    frozen_names = ["H1_I", "Hsym_lower"] + (["U"] if freeze_u else [])
    frozen = {name: p[name] for name in frozen_names}
    print(f"{label}: epochs={epochs} rho={rho:g} freeze_u={freeze_u}", flush=True)
    solution = Train(
        generating_set="Qaggle",
        method="DF-THC-SOS",
        epochs=int(epochs),
        learning_rate=float(learning_rate),
        learning_rate_final=float(learning_rate_final),
        rho=float(rho),
        params=p,
        ref=REFERENCE_DATA,
        print_interval=max(1000, int(epochs) // 5),
        epsilon_dfthc=0.008,
        frozen_params=frozen,
        lambda_floor=49.0,
        sos_gap_floor=3.914898,
    )
    result = native(solution["params_best"])
    if freeze_u:
        result["U"] = np.asarray(params["U"]).copy()
    return result, {name: float(value) for name, value in solution["values"].items()
                    if np.ndim(value) == 0}


def canonicalize_words(words, edges):
    words = np.asarray(words, dtype=np.int64)
    vectors = reconstruct_u_vectors(words / 512, edges)
    remaining = vectors.copy()
    angles = np.zeros_like(words, dtype=np.float64)
    for j in reversed(range(len(edges))):
        parent, child = edges[j]
        norm = np.hypot(remaining[:, parent], remaining[:, child])
        sign = np.where(np.abs(remaining[:, parent]) < 1e-12,
                        np.where(remaining[:, child] > 0, -1.0, 1.0),
                        np.where(remaining[:, parent] < 0, -1.0, 1.0))
        angles[:, j] = np.where(
            norm < 1e-12, 0.0,
            np.arctan2(remaining[:, child] * sign, remaining[:, parent] * sign),
        )
        remaining[:, parent] = np.where(norm < 1e-12, 0.0, norm * sign)
        remaining[:, child] = 0.0
    signed = np.rint(angles * (512 / np.pi)).astype(np.int64) % 1024
    if np.any(((signed >> 9) ^ (signed >> 8)) & 1):
        raise ArithmeticError("signed ten-bit gauge failed")
    decoded = reconstruct_u_vectors(signed / 512, edges)
    row_sign = np.where(np.sum(vectors * decoded, axis=1) < 0, -1.0, 1.0)
    if float(np.max(np.abs(decoded * row_sign[:, None] - vectors))) > 1e-10:
        raise ArithmeticError("signed gauge changed a decoded vector")
    return signed, decoded


def encode_payload(params):
    params = native(params)
    kit_inputs = kit_load_inputs("FeMoCo76o113e", params)
    words = np.asarray(encode_inputs(kit_inputs, TREE_EDGES, brot=10, rounding="nearest"),
                       dtype=np.int64)
    words, decoded = canonicalize_words(words, TREE_EDGES)
    shape = params["W"].shape
    params["U"] = decoded[76:].reshape(shape[0], shape[1], 76)
    return params, words


def exact_metrics(params, words):
    inputs = VC.load_inputs(params)
    metrics = VC.evaluate_words(inputs, words, TREE_EDGES, 10)
    return inputs, metrics


def build_candidate(name, params, words):
    out = pathlib.Path("/tmp/qcc_v7_candidates") / name
    shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True)
    circuits, reports = [], []
    for index, delta in enumerate(batch_deltas(words, 3)):
        delta = np.asarray(delta, dtype=np.int64)
        if len(delta) % 4:
            delta = np.pad(delta, ((0, 4 - len(delta) % 4), (0, 0)))
        circuit, report = FOUR.build(delta, brot=10, return_report=True)
        if circuit.num_qubits > 128:
            raise ValueError(f"{name}: ANGLE_{index} uses {circuit.num_qubits} qubits")
        circuit.to_file(str(out / f"angle_k_{index}.kmb"), format="kmb")
        circuits.append(circuit)
        reports.append(report)
    payload = {key: np.asarray(value) for key, value in native(params).items()}
    payload["tree_edges"] = TREE_EDGES
    payload["angle_words"] = words
    np.savez(out / "submission.npz", **payload)
    inputs, metrics = exact_metrics(payload, words)
    angle_toffoli = 2 * sum(c.operation_statistics().num_toffoli_like for c in circuits)
    resources = VC.resource_costs(inputs, circuits, 10, angle_toffoli)
    score = VC.score_from_metrics(metrics, resources)
    return out, metrics, resources, float(score), reports


def serializable(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {key: serializable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [serializable(item) for item in value]
    return value


parents = {"baseline": native(BASE_PARAMETERS)}
TRAINING_REPORT = {
    "preset": TRAIN_PRESET,
    "incumbent_chemistry_cost": INCUMBENT_CHEM_COST,
    "base_offset_refit": BASE_OFFSET_REPORT,
    "trials": [],
}
accepted_candidates = []

for spec in SEARCH_PLAN:
    name, shape = spec["name"], tuple(spec["shape"])
    print(f"\n=== {name}: target R/B/C={shape} from {spec['parent']} ===", flush=True)
    trial = {"name": name, "shape": list(shape), "parent": spec["parent"]}
    if spec["parent"] not in parents:
        trial["skipped"] = f"parent {spec['parent']} did not finish"
        TRAINING_REPORT["trials"].append(trial)
        print(f"{name}: skipped because parent {spec['parent']} is unavailable", flush=True)
        continue
    try:
        parent = parents[spec["parent"]]
        pruned, pruning = DT.prune_dimensions(parent, *shape)
        recovery_epochs, lambda_epochs = spec["epochs"]
        trained, recovery_values = train_phase(
            pruned, epochs=recovery_epochs, rho=1e-6,
            learning_rate=3e-4, learning_rate_final=5e-5,
            label=f"{name}/accuracy",
        )
        trained, lambda_values = train_phase(
            trained, epochs=lambda_epochs, rho=1e-5,
            learning_rate=1e-4, learning_rate_final=1e-5,
            label=f"{name}/lambda",
        )
        parents[name] = trained
        np.savez(WORK / f"v7_continuous_{name}.npz", **trained)

        payload, words = encode_payload(trained)
        _, before_metrics = exact_metrics(payload, words)
        quantized_refit = None
        # Refit W/H2_I with the quantized vectors fixed. If a starter-kit
        # revision cannot freeze U, retain the already evaluated candidate.
        try:
            payload, quantized_refit = train_phase(
                payload, epochs=max(6000, lambda_epochs // 2), rho=1e-5,
                learning_rate=5e-5, learning_rate_final=5e-6,
                freeze_u=True, label=f"{name}/fixed-word-refit",
            )
        except Exception as exc:
            quantized_refit = {"skipped": type(exc).__name__, "message": str(exc)}
            print(f"{name}: fixed-word trainer refit skipped: {exc}", flush=True)

        _, refit_metrics = exact_metrics(payload, words)
        if VC.passes_accuracy(refit_metrics):
            payload, offset_report = OR.refit_offsets(
                payload, words, TREE_EDGES, 10,
                radii=(2e-4, 6e-4, 2e-3), maxiter=50,
            )
        else:
            offset_report = {"improved": False, "skipped": "pre-offset metrics rejected"}
        _, final_metrics = exact_metrics(payload, words)

        trial.update(
            pruning=pruning,
            recovery_values=recovery_values,
            lambda_values=lambda_values,
            metrics_before_fixed_refit=serializable(before_metrics),
            fixed_word_refit=quantized_refit,
            offset_refit=offset_report,
            final_metrics=serializable(final_metrics),
            numerical_pass=bool(VC.passes_accuracy(final_metrics)),
        )
        print(
            f"{name}: e1={1e3*final_metrics['error_1']:.3f}mHa "
            f"e2={1e3*final_metrics['error_2']:.3f}mHa "
            f"lambda={final_metrics['lambda_eff']:.5f} "
            f"accepted={trial['numerical_pass']}", flush=True,
        )
        if trial["numerical_pass"]:
            out, metrics, resources, cost, build_reports = build_candidate(name, payload, words)
            trial.update(
                resources=serializable(resources),
                exact_projected_cost=cost,
                beats_incumbent=bool(cost < INCUMBENT_CHEM_COST),
                build_reports=serializable(build_reports),
            )
            print(
                f"{name}: Q={resources['qubits']} T={resources['toffoli']:.0f} "
                f"cost={cost:.8g} beats incumbent={cost < INCUMBENT_CHEM_COST}",
                flush=True,
            )
            if cost < INCUMBENT_CHEM_COST:
                accepted_candidates.append((cost, name, out))
    except Exception as exc:
        trial.update(error_type=type(exc).__name__, error=str(exc))
        print(f"{name}: experiment failed safely: {type(exc).__name__}: {exc}", flush=True)
    TRAINING_REPORT["trials"].append(trial)
    (WORK / "v7_training_report.partial.json").write_text(
        json.dumps(serializable(TRAINING_REPORT), indent=2, default=str) + "\n"
    )

SELECTED_CHEMISTRY = "incumbent"
SELECTED_CHEMISTRY_COST = float(INCUMBENT_CHEM_COST)
if accepted_candidates:
    SELECTED_CHEMISTRY_COST, SELECTED_CHEMISTRY, selected_dir = min(accepted_candidates)
    shutil.rmtree(SUB / "dfthcsos")
    shutil.copytree(selected_dir, SUB / "dfthcsos")
    print(f"Selected {SELECTED_CHEMISTRY}: {INCUMBENT_CHEM_COST:.8g} -> "
          f"{SELECTED_CHEMISTRY_COST:.8g}", flush=True)
else:
    print("No accepted structural candidate beat the incumbent; retaining it", flush=True)

TRAINING_REPORT["selected"] = SELECTED_CHEMISTRY
TRAINING_REPORT["selected_chemistry_cost"] = SELECTED_CHEMISTRY_COST
(WORK / "v7_training_report.json").write_text(
    json.dumps(serializable(TRAINING_REPORT), indent=2, default=str) + "\n"
)

# Release trainer state before the full simulators allocate their worker pools.
parents.clear()
accepted_candidates.clear()
del REFERENCE_DATA, BASE_CIRCUITS, BASE_ARCHIVE, base_inputs, base_quick
try:
    jax.clear_caches()
except Exception:
    pass
import gc
gc.collect()


In [ ]:
def with_heartbeat(label, function, every=60):
    finished = threading.Event()
    started = time.monotonic()
    def heartbeat():
        while not finished.wait(every):
            print(f"    {label} still running ({(time.monotonic()-started)/60:.1f} min)", flush=True)
    monitor = threading.Thread(target=heartbeat, daemon=True)
    monitor.start()
    try:
        return function()
    finally:
        finished.set()
        monitor.join()


started = time.monotonic()
print("[1/3] Full inverse-square-root verification", flush=True)
from invsqrt import verify_submission_invsqrt as VI
invsqrt_result = with_heartbeat(
    "inverse-square-root verifier",
    lambda: VI.verify(load_circuit(SUB / "invsqrt.kmb"),
                      VI.ProblemSpecInvSqrt.for_n_p(10), max_workers=WORKERS),
)
assert invsqrt_result["status"] == "PASS" and invsqrt_result["verified_shots"] == 200540928
assert invsqrt_result["num_qubits"] == 82

print("[2/3] Full DQI verification", flush=True)
from dqi import verify_submission_dqi as VD
dqi_circuit = load_circuit(SUB / "dqi.kmb")
dqi_trials = []
for salt in ("", "independent-v7-a", "independent-v7-b", "independent-v7-c"):
    result = with_heartbeat(
        f"DQI verifier salt={salt or '<empty>'}",
        lambda salt=salt: VD.verify(dqi_circuit, VD.ProblemSpecDQI.benchmark_instance(),
                                    salt=salt, max_workers=WORKERS),
    )
    assert result["status"] == "PASS" and result["verified_shots"] == 102400
    assert result["num_qubits"] == 898
    dqi_trials.append({"salt": salt, **result})
    print(f"    salt={salt or '<empty>'}: value={result['value_errors']} "
          f"phase={result['phase_errors']} R={result['repetition_factor']:.6f} "
          f"cost={result['score']:.7g}", flush=True)
dqi_result = dqi_trials[0]
dqi_worst_result = max(dqi_trials, key=lambda result: result["score"])

print("[3/3] Full DFTHC-SOS verification", flush=True)
chem_paths = sorted((SUB / "dfthcsos").glob("angle_k_*.kmb"),
                    key=lambda path: int(path.stem.rsplit("_", 1)[1]))
circuits = [load_circuit(path) for path in chem_paths]
with np.load(SUB / "dfthcsos" / "submission.npz", allow_pickle=False) as archive:
    factorization = {name: np.array(archive[name]) for name in archive.files}
tree_edges = factorization.pop("tree_edges")
shape = tuple(map(int, factorization["W"].shape))
dfthcsos_result = with_heartbeat(
    "chemistry verifier", lambda: VC.verify(circuits, factorization, tree_edges),
)
assert dfthcsos_result["status"] == "PASS"
assert dfthcsos_result["verified_shots"] == (76 + shape[0] * shape[1]) * 64

RESULTS = {"invsqrt": invsqrt_result, "dqi": dqi_result, "dfthcsos": dfthcsos_result}
SCORE = math.prod(REFERENCE[name] / RESULTS[name]["score"] for name in REFERENCE) ** (1 / 3)
WORST_SALT_SCORE = (
    (REFERENCE["invsqrt"] / invsqrt_result["score"])
    * (REFERENCE["dqi"] / dqi_worst_result["score"])
    * (REFERENCE["dfthcsos"] / dfthcsos_result["score"])
) ** (1 / 3)
if min(SCORE, WORST_SALT_SCORE) < 10.8:
    raise AssertionError(f"v7 fell below safety floor: {SCORE:.6f}, {WORST_SALT_SCORE:.6f}")

rows = []
for name, label in (("invsqrt", "Inverse square root"), ("dqi", "DQI decoder"),
                    ("dfthcsos", f"FeMoCo {shape}")):
    result = RESULTS[name]
    rows.append({
        "Task": label,
        "Qubits": result["num_qubits"],
        "Toffoli": f"{result['toffoli_count']:,.3f}",
        "Cost": f"{result['score']:.7g}",
        "Reference / cost": f"{REFERENCE[name] / result['score']:.3f}",
    })
display(pd.DataFrame(rows).style.hide(axis="index"))
print(f"Selected chemistry: {SELECTED_CHEMISTRY}")
print(f"Full verification completed in {(time.monotonic()-started)/60:.1f} min")
print(f"Verified local score: {SCORE:.5f}")
print(f"Worst reviewed-salt score: {WORST_SALT_SCORE:.5f}")


In [ ]:
MAX_ID = 2**30 // 1536
problems = []
all_files = sorted(path.relative_to(SUB).as_posix() for path in SUB.rglob("*") if path.is_file())
angles = sorted(
    (name for name in all_files if name.startswith("dfthcsos/angle_k_") and name.endswith(".kmb")),
    key=lambda name: int(pathlib.Path(name).stem.rsplit("_", 1)[1]),
)
expected = {"invsqrt.kmb", "dqi.kmb", "dfthcsos/submission.npz"} | set(angles)
if set(all_files) != expected:
    problems.append(f"manifest differs: {all_files}")
if [int(pathlib.Path(name).stem.rsplit("_", 1)[1]) for name in angles] != list(range(len(angles))):
    problems.append("chemistry circuit indices are not contiguous")
for name in sorted(item for item in all_files if item.endswith(".kmb")):
    circuit = load_circuit(SUB / name)
    top_id = max(circuit.num_qubits, circuit.num_bits) - 1
    unnamed = [register for register in circuit.register_data if not isinstance(register, str)]
    if top_id >= MAX_ID:
        problems.append(f"{name}: largest ID {top_id} >= {MAX_ID}")
    if unnamed:
        problems.append(f"{name}: unnamed registers {unnamed}")
with np.load(SUB / "dfthcsos" / "submission.npz", allow_pickle=False) as archive:
    for name in archive.files:
        value = archive[name]
        expected_kind = np.integer if name in {"tree_edges", "angle_words"} else np.floating
        little = value.dtype.byteorder in {"<", "|"} or (
            value.dtype.byteorder == "=" and sys.byteorder == "little")
        if not np.issubdtype(value.dtype, expected_kind) or not little:
            problems.append(f"submission.npz {name}: invalid dtype {value.dtype.str}")
if problems:
    raise AssertionError("\n".join(problems))

ZIP_PATH = WORK / "submission.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(SUB.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(SUB).as_posix())
with zipfile.ZipFile(ZIP_PATH) as archive:
    infos = archive.infolist()
    if {info.filename for info in infos} != expected or archive.testzip() is not None:
        raise AssertionError("ZIP validation failed")
    member_hashes = {info.filename: hashlib.sha256(archive.read(info.filename)).hexdigest()
                     for info in infos}

summary = {
    "score": SCORE,
    "worst_reviewed_salt_score": WORST_SALT_SCORE,
    "selected_chemistry": SELECTED_CHEMISTRY,
    "results": RESULTS,
    "dqi_salt_trials": dqi_trials,
    "training_report": TRAINING_REPORT,
    "source_plan": PLAN,
    "member_sha256": member_hashes,
    "archive_sha256": sha256_file(ZIP_PATH),
    "archive_bytes": ZIP_PATH.stat().st_size,
}
(WORK / "summary_v7.json").write_text(json.dumps(summary, indent=2, default=str) + "\n")
(WORK / "submission-members-v7.sha256").write_text(
    "".join(f"{member_hashes[name]}  {name}\n" for name in sorted(member_hashes))
)
print(f"Submission ready: {ZIP_PATH}")
print(f"Selected        : {SELECTED_CHEMISTRY}")
print(f"Archive SHA-256 : {summary['archive_sha256']}")
print(f"Verified score  : {SCORE:.5f}")
ZIP_PATH


## How to read the result

The decisive lines are `Selected chemistry`, `Verified local score`, and the
final full-verifier table. A rejected training branch is recorded in
`v7_training_report.json` and does not affect the submission. Continuous
trainer loss is never used as acceptance evidence.

The output directory also contains every continuous checkpoint, the partial
report written after each branch, `summary_v7.json`, the member hash manifest,
and `submission.zip`. This makes a long run resumable and preserves enough
evidence to decide which shape should receive a longer follow-up training run.
